# Node-level Keyword Extraction (Reviews)

This notebook experiments with extracting **top-N representative keywords or short phrases** for each node, split by **positive vs negative** sentiment.

Goals:
- Filter to clearly positive/negative reviews (configurable thresholds).
- Language-aware preprocessing:
  - English: tokenization, lowercasing, stopword removal.
  - Korean: morphological analysis with noun/adjective extraction (falls back to simple tokenization if a tagger is unavailable).
- Candidate keywords via:
  - frequency counts
  - optional TF-IDF (node-level)
  - optional bi-grams
- Output ranked keyword lists per node and polarity.

Notes:
- This is designed for clarity/experimentation, not production optimization.
- Use the outputs later to pick representative reviews.


In [1]:
# Core imports
import os
import re
import sqlite3
from pathlib import Path
from collections import Counter

import pandas as pd


In [2]:
# ---- Config ----
DB_PATH = Path("../data/app.sqlite")
MECAB_PATH = "/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic"

# Sentiment filtering
SENTIMENT_SCALE = "auto"  # "auto", "zero_to_one", or "minus_one_to_one"
POS_THRESHOLD_0_1 = 0.6
NEG_THRESHOLD_0_1 = 0.4
POS_THRESHOLD_M11 = 0.2
NEG_THRESHOLD_M11 = -0.2
MIN_CONFIDENCE = 0.0  # set >0 to require confidence (e.g., 0.6)

# Keyword extraction
TOP_N = 3
USE_TFIDF = True
USE_BIGRAMS = True
MIN_TOKEN_LEN = 2  # applies to English; Korean uses a separate filter

# If your sentiments are stored by multiple methods/versions, select them here.
SENTIMENT_METHOD = None  # e.g., "vader", or None to keep all
SENTIMENT_VERSION = None  # e.g., "1.0", or None to keep all


In [3]:
# ---- Load reviews + sentiment ----
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DB not found: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

query = '''
SELECT
  r.id AS review_id,
  r.node_id,
  n.name AS node_name,
  r.content AS text,
  r.entity_name,
  r.created_at,
  s.sentiment_raw,
  s.confidence,
  s.method,
  s.version
FROM review r
LEFT JOIN nodes n ON n.id = r.node_id
JOIN review_sentiment s ON s.review_id = r.id
WHERE r.node_id IS NOT NULL
'''

params = {}
if SENTIMENT_METHOD is not None:
    query += " AND s.method = :method"
    params["method"] = SENTIMENT_METHOD
if SENTIMENT_VERSION is not None:
    query += " AND s.version = :version"
    params["version"] = SENTIMENT_VERSION

reviews = pd.read_sql_query(query, conn, params=params)
conn.close()

reviews.head()


,review_id,node_id,node_name,text,entity_name,created_at,sentiment_raw,confidence,method,version
0,1,1,쏘마치 (BHC),BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...,BHC 쏘마치,2025-12-26T09:39:18.074Z,0.530003,0.530003,model,cardiffnlp/twitter-xlm-roberta-base-sentiment
1,2,2,K마라치킨 (처갓집 양념치킨),처가집 양념통닭에서 마라치킨 주문. 그냥 달달한 양념 치킨에 마라향 살짝 추가한 정...,처가집 양념통닭 K마라치킨,2025-12-29T07:53:41.320Z,0.000000,0.379968,model,cardiffnlp/twitter-xlm-roberta-base-sentiment
2,3,103,멕시카나,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...,멕시카나,2025-12-29T07:54:49.553Z,-0.635604,0.635604,model,cardiffnlp/twitter-xlm-roberta-base-sentiment
3,4,10,맵소디 (BBQ),비비큐에서 맵소디 배달 주문. 양념 맛은 기억나지 않지만 소스 안쪽에서도 바삭한 식...,비비큐 맵소디,2025-12-29T07:58:38.948Z,0.000000,0.572736,model,cardiffnlp/twitter-xlm-roberta-base-sentiment
4,5,6,노랑통닭,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,노랑통닭,2025-12-29T07:59:37.482Z,0.694385,0.694385,model,cardiffnlp/twitter-xlm-roberta-base-sentiment


## 1) Filter to clearly positive/negative reviews

We detect the sentiment scale and apply conservative thresholds:
- For 0..1 scores: positive ≥ **0.6**, negative ≤ **0.4**
- For -1..1 scores: positive ≥ **0.2**, negative ≤ **-0.2**

Adjust the thresholds above if needed. Optionally apply a minimum confidence.


In [4]:
def detect_scale(series, mode="auto"):
    if mode == "zero_to_one":
        return "zero_to_one"
    if mode == "minus_one_to_one":
        return "minus_one_to_one"

    # auto
    min_v = series.min()
    max_v = series.max()
    if min_v >= -1.1 and max_v <= 1.1 and (min_v < 0):
        return "minus_one_to_one"
    return "zero_to_one"


def label_sentiment(score, scale):
    if scale == "minus_one_to_one":
        if score >= POS_THRESHOLD_M11:
            return "positive"
        if score <= NEG_THRESHOLD_M11:
            return "negative"
        return None
    # zero_to_one
    if score >= POS_THRESHOLD_0_1:
        return "positive"
    if score <= NEG_THRESHOLD_0_1:
        return "negative"
    return None

scale = detect_scale(reviews["sentiment_raw"], SENTIMENT_SCALE)

filtered = reviews.copy()
if MIN_CONFIDENCE > 0:
    filtered = filtered[filtered["confidence"].fillna(0) >= MIN_CONFIDENCE]

filtered["polarity"] = filtered["sentiment_raw"].apply(lambda s: label_sentiment(s, scale))
filtered = filtered[filtered["polarity"].notna()].reset_index(drop=True)

print("scale:", scale)
print("reviews kept:", len(filtered), "of", len(reviews))
filtered["polarity"].value_counts()


scale: minus_one_to_one
reviews kept: 44 of 54


polarity
positive    27
negative    17
Name: count, dtype: int64

## 2) Language-aware preprocessing

English:
- tokenize by letters/numbers
- lowercase
- remove stopwords

Korean:
- prefer morphological analysis (Okt) to extract nouns/adjectives
- fallback to simple Hangul tokenization if tagger is unavailable


In [5]:
# --- English preprocessing ---

import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
EN_STOPWORDS = set(stopwords.words("english"))


# Customizable stopwords
EN_STOPWORDS_CUSTOM = {
    "ok","okay","yeah","yep","nope","hi","hello","thanks","thank","pls","please",
    "maybe","just","like","really","very","thing","stuff"
}
EN_STOPWORDS = EN_STOPWORDS.union(EN_STOPWORDS_CUSTOM)

EN_TOKEN_RE = re.compile(r"[a-zA-Z0-9']+")


def tokenize_english(text):
    tokens = [t.lower() for t in EN_TOKEN_RE.findall(text or "")]
    tokens = [t for t in tokens if t not in EN_STOPWORDS and len(t) >= MIN_TOKEN_LEN]
    return tokens


# --- Korean preprocessing ---
# POS allowlist: explicit + configurable
KOREAN_POS_ALLOW = {"NNG", "NNP", "VA"}

# Customizable stopwords (add as needed)
KOREAN_STOPWORDS = {
    "근데","그런데","같다","이건","저건","원래","무슨","진짜","정말",
    "그냥","좀","조금","항상","대박","완전","너무","약간","그리고",
    "하지만","그래서","왜","뭔가","그거","이거","저거","여기","거기",
    "저기","이렇게","그렇게","저렇게","있다","없다","하다","되다"
}

# Simple canonicalization map (extend as needed)
KOREAN_CANON = {
    "맛나다": "맛있다",
    "맛있음": "맛있다",
    "맛있어요": "맛있다",
}

from konlpy.tag import Mecab
mecab = Mecab(dicpath=MECAB_PATH)


def tokenize_korean(text):
    tokens = []
    for word, pos in mecab.pos(text or ""):
        if pos not in KOREAN_POS_ALLOW:
            continue
        if len(word) < 2:
            continue
        # Normalize to canonical forms where applicable
        canon = KOREAN_CANON.get(word, word)
        if canon in KOREAN_STOPWORDS:
            continue
        tokens.append(canon)
    return tokens

KOREAN_TOKENIZER = "mecab"


def detect_language(text):
    # simple heuristic: Hangul => Korean
    return "ko" if re.search(r"[가-힣]", text or "") else "en"


def tokenize(text):
    lang = detect_language(text)
    if lang == "ko":
        return tokenize_korean(text)
    return tokenize_english(text)

print("korean tokenizer:", KOREAN_TOKENIZER)


korean tokenizer: mecab


In [6]:
# ---- Explicit tokenization step (single source of truth) ----
filtered["tokens"] = filtered["text"].apply(tokenize)
filtered["tokens_joined"] = filtered["tokens"].apply(lambda t: " ".join(t))

filtered[["text", "tokens"]].head()


,text,tokens
0,BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...,"[서현, 시범, 근처, 지점, 배달, 최수, 주문가, 주문, 마치, 주문, 배달, ..."
1,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...,"[배달, 멕시카나, 치킨, 할인, 주문, 금점, 영업, 할인, 의미]"
2,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,"[노랑, 통닭, 살도, 가격, 비싸, 튀김, 부스러기, 과자, 맛있, 치킨, 프라이..."
3,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...,"[유명, 쉐프, 협업, 풍치, 주문, 소스, 느낌, 치킨, 치킨, 최악]"
4,굽네치킨에서 유일하게 좋아하는 고추바사삭. 근데 살이 너무 없는것 같다,"[치킨, 유일, 고추]"


## 3) Candidate keyword extraction

We create per-review tokens, then aggregate per node and polarity.
Optional bigrams are appended when `USE_BIGRAMS=True`.


In [7]:
def add_bigrams(tokens):
    return [f"{a} {b}" for a, b in zip(tokens, tokens[1:])]


def consolidate_phrases(tokens, min_count=2):
    # Prefer meaningful bi-grams when frequent.
    # Avoid overlapping phrases by taking the longest matches first.
    if len(tokens) < 2:
        return tokens

    bigrams = add_bigrams(tokens)
    bigram_counts = Counter(bigrams)
    frequent = {bg for bg, c in bigram_counts.items() if c >= min_count}

    consolidated = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1:
            bg = f"{tokens[i]} {tokens[i+1]}"
            if bg in frequent:
                consolidated.append(bg)
                i += 2
                continue
        consolidated.append(tokens[i])
        i += 1

    return consolidated


def sentiment_weight(confidence):
    # Weight keyword contributions by sentiment confidence.
    # Ambiguous reviews get lower weight.
    if confidence is None:
        return 1.0
    try:
        c = float(confidence)
    except Exception:
        return 1.0
    # squashing: 0.0 -> 0.5, 1.0 -> 1.5
    return 0.5 + c


def preprocess_reviews(df):
    all_tokens = []
    all_weights = []

    for toks, conf in zip(df["tokens"], df["confidence"].fillna(0)):
        if USE_BIGRAMS and len(toks) >= 2:
            toks = consolidate_phrases(toks)
        all_tokens.append(toks)
        all_weights.append(sentiment_weight(conf))

    out = df.copy()
    out["tokens"] = all_tokens
    out["tokens_joined"] = [" ".join(toks) for toks in all_tokens]
    out["weight"] = all_weights
    return out

processed = preprocess_reviews(filtered)
processed.head()


,review_id,node_id,node_name,text,entity_name,created_at,sentiment_raw,confidence,method,version,polarity,tokens,tokens_joined,weight
0,1,1,쏘마치 (BHC),BHC서현시범단지점. 근처 지점중 배달 최수 주문가가 가장 저렴해 치킨 주문. 쏘마...,BHC 쏘마치,2025-12-26T09:39:18.074Z,0.530003,0.530003,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,positive,"[서현, 시범, 근처, 지점, 배달, 최수, 주문가, 주문, 마치, 주문 배달, 마...",서현 시범 근처 지점 배달 최수 주문가 주문 마치 주문 배달 마늘 토핑 추가 주문 ...,1.030003
1,3,103,멕시카나,배달앱에서 멕시카나 치킨 할인한다기에 주문하려했더니 미금점 영업을 하지 않는다. 이...,멕시카나,2025-12-29T07:54:49.553Z,-0.635604,0.635604,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,negative,"[배달, 멕시카나, 치킨, 할인, 주문, 금점, 영업, 할인, 의미]",배달 멕시카나 치킨 할인 주문 금점 영업 할인 의미,1.135604
2,5,6,노랑통닭,노랑통닭은 순살도 가격이 더 비싸지 않아 좋다. 튀김 부스러기는 과자같이 바삭하니 ...,노랑통닭,2025-12-29T07:59:37.482Z,0.694385,0.694385,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,positive,"[노랑, 통닭, 살도, 가격, 비싸, 튀김, 부스러기, 과자, 맛있, 치킨, 프라이...",노랑 통닭 살도 가격 비싸 튀김 부스러기 과자 맛있 치킨 프라이 냄새,1.194385
3,6,57,푸라닭,푸라닭에서 유명 쉐프와 협업했다는 깐풍치킨 주문. 소스가 볶은 느낌이 아니라 흥건하...,푸라닭,2025-12-29T08:35:01.386Z,0.643638,0.643638,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,positive,"[유명, 쉐프, 협업, 풍치, 주문, 소스, 느낌, 치킨, 치킨, 최악]",유명 쉐프 협업 풍치 주문 소스 느낌 치킨 치킨 최악,1.143638
4,9,20,굽네치킨,굽네치킨에서 유일하게 좋아하는 고추바사삭. 근데 살이 너무 없는것 같다,굽네치킨,2025-12-29T09:12:47.282Z,-0.496433,0.496433,model,cardiffnlp/twitter-xlm-roberta-base-sentiment,negative,"[치킨, 유일, 고추]",치킨 유일 고추,0.996433


In [8]:
# ---- Frequency-based keywords (weighted) ----

def keyword_counts(df, top_n=TOP_N):
    results = []
    for (node_id, node_name, polarity), group in df.groupby(["node_id", "node_name", "polarity"], dropna=False):
        counter = Counter()
        source_counter = Counter()
        for toks, w in zip(group["tokens"], group["weight"]):
            if not toks:
                continue
            # count once per review for source count
            source_counter.update(set(toks))
            # weighted frequency
            counter.update({t: w for t in toks})
        for token, score in counter.most_common(top_n):
            results.append({
                "node_id": node_id,
                "node_name": node_name,
                "polarity": polarity,
                "keyword": token,
                "score": float(score),
                "source_count": int(source_counter.get(token, 0)),
                "source": "count",
            })
    return pd.DataFrame(results)

count_df = keyword_counts(processed)
count_df.head()


,node_id,node_name,polarity,keyword,score,source_count,source
0,1,쏘마치 (BHC),positive,서현,1.030003,1,count
1,1,쏘마치 (BHC),positive,시범,1.030003,1,count
2,1,쏘마치 (BHC),positive,근처,1.030003,1,count
3,6,노랑통닭,negative,치킨,1.165789,1,count
4,6,노랑통닭,positive,맛있,2.258784,2,count


In [9]:
# ---- Optional TF-IDF per node (per polarity) ----
if USE_TFIDF:
    from sklearn.feature_extraction.text import TfidfVectorizer

    def keyword_tfidf(df, top_n=TOP_N):
        results = []
        for polarity, pgroup in df.groupby("polarity"):
            # Each node is treated as a single document
            docs = []
            node_keys = []
            node_token_sources = []

            for (node_id, node_name), ngroup in pgroup.groupby(["node_id", "node_name"], dropna=False):
                node_keys.append((node_id, node_name))
                docs.append(" ".join(ngroup["tokens_joined"].tolist()))

                # source count per token (reviews containing the token)
                sc = Counter()
                for toks in ngroup["tokens"]:
                    sc.update(set(toks))
                node_token_sources.append(sc)

            if not docs:
                continue

            vectorizer = TfidfVectorizer(token_pattern=r"[^ ]+", lowercase=False)
            tfidf = vectorizer.fit_transform(docs)
            vocab = vectorizer.get_feature_names_out()

            for i, (node_id, node_name) in enumerate(node_keys):
                row = tfidf.getrow(i)
                if row.nnz == 0:
                    continue
                top_idx = row.toarray().ravel().argsort()[-top_n:][::-1]
                for idx in top_idx:
                    score = row[0, idx]
                    if score <= 0:
                        continue
                    token = vocab[idx]
                    results.append({
                        "node_id": node_id,
                        "node_name": node_name,
                        "polarity": polarity,
                        "keyword": token,
                        "score": float(score),
                        "source_count": int(node_token_sources[i].get(token, 0)),
                        "source": "tfidf",
                    })

        return pd.DataFrame(results)

    tfidf_df = keyword_tfidf(processed)
else:
    tfidf_df = pd.DataFrame()

(tfidf_df.head() if not tfidf_df.empty else tfidf_df)


,node_id,node_name,polarity,keyword,score,source_count,source
0,6,노랑통닭,negative,치킨,1.000000,1,tfidf
1,14,땡겨요,negative,비싸,0.707107,1,tfidf
2,14,땡겨요,negative,달비,0.707107,1,tfidf
3,20,굽네치킨,negative,이상,0.516538,1,tfidf
4,20,굽네치킨,negative,이질,0.333288,1,tfidf


## 4) Combine results and display

We show a readable table per node and polarity. You can choose which source to display (`count` or `tfidf`).


In [10]:
def combine_results(count_df, tfidf_df):
    frames = [count_df]
    if not tfidf_df.empty:
        frames.append(tfidf_df)
    return pd.concat(frames, ignore_index=True)

results = combine_results(count_df, tfidf_df)
results.head()


,node_id,node_name,polarity,keyword,score,source_count,source
0,1,쏘마치 (BHC),positive,서현,1.030003,1,count
1,1,쏘마치 (BHC),positive,시범,1.030003,1,count
2,1,쏘마치 (BHC),positive,근처,1.030003,1,count
3,6,노랑통닭,negative,치킨,1.165789,1,count
4,6,노랑통닭,positive,맛있,2.258784,2,count


In [11]:
results['keyword'].unique()

array(['서현', '시범', '근처', '치킨', '맛있', '노랑', '통닭', '달비', '비싸', '볼케이노', '소스',
       '이상', '족발', '배달', '막국수', '속초', '시장', '구경', '엄지', '발가락', '동네', '며칠',
       '니자', '발뒤꿈치', '정자동', '정동', '마트', '콩나물', '두부', '이마트', '홈플러스',
       '와인 주문', '맥주', '재고', '확인', '매운맛', '교촌', '레드', '구성', '냄새', '국산',
       '유명', '쉐프', '협업', '튀김옷', '닭고기', '양념', '과자', '크림', '저녁', '커피', '맛집',
       '풍자', '유튜브', '유명해', '세일', '구입', '고기', '조리법', '할인', '부대', '전골',
       '건더기', '아메리카노', '중간', '사이즈', '짬뽕', '단대오거리역', '점심', '음식', '친절',
       '멕시카나', '가격', '가쓰오부시', '새해', '첫날', '영업', '비엔나', '주부', '식감', '정형',
       '외과', '민생', '지원금', '성남', '현대', '페치', '블랙', '페퍼', '고추', '광어회', '제주',
       '이질', '표현', '정작', '얼떨결', '대부분', '취소', '배송', '와인', '이번', '문제', '동원',
       '의미', '금점', '부위', '저번', '필요', '광어', '참치', '주문', '마늘', '부스러기',
       '닭강정', '동해', '의사', '주사', '정형외과', '프라이스', '초보', '프로', '나중', '풍치',
       '한잔', '테라스', '삼겹살', '추가', '치방', '프리미엄', '몰츠', '국물', '해장', '타면',
       '덕분', '생각', '일반', '생물', '마지막', '갈비', '육회', '조합', '할머니'

In [12]:
def show_node_tables(results_df, source="count", top_n=TOP_N):
    view = results_df[results_df["source"] == source].copy()
    for (node_id, node_name), group in view.groupby(["node_id", "node_name"], dropna=False):
        title = f"Node {node_id} — {node_name}" if pd.notna(node_name) else f"Node {node_id}"
        print("=" * len(title))
        print(title)
        print("=" * len(title))
        for polarity, pgroup in group.groupby("polarity"):
            print(f"\n[{polarity}]")
            display(pgroup.sort_values("score", ascending=False).head(top_n)
                    [["keyword", "score", "source_count"]].reset_index(drop=True))
        print("\n")

show_node_tables(results, source="count", top_n=TOP_N)


Node 1 — 쏘마치 (BHC)

[positive]


,keyword,score,source_count
0,서현,1.030003,1
1,시범,1.030003,1
2,근처,1.030003,1




Node 6 — 노랑통닭

[negative]


,keyword,score,source_count
0,치킨,1.165789,1



[positive]


...

,keyword,score,source_count
0,고추,1.263275,1
1,치킨,1.263275,1
2,할인,1.263275,1




Node 120 — 이마트 회

[negative]


,keyword,score,source_count
0,세일,2.310143,2
1,구입,2.310143,2
2,저녁,1.187139,1



[positive]


,keyword,score,source_count
0,광어회,2.117773,2
1,제주,1.140952,1
2,할인,1.140952,1


## 5) Next steps (optional)

- Tune thresholds, confidence filter, or sentiment method/version.
- Try language-specific stopword lists or domain vocab.
- Use the keyword lists to select representative reviews per node/polarity.
